# Exploratory Data Analysis

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import scipy as sp
import statsmodels as sm
import sklearn as sk
import os
import re
import time
from tqdm import tqdm
import math
import time

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac

In [ ]:
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_reference_data
%store -r restaurant_sales_data

## Static Reference Data Exploration

### Retrieval

Retrieve the four static reference data frames

In [ ]:
# Promotional items (30 rows)
before_after_details = static_reference_data['before_after_details'].copy()

# Specific customer information: for matching customers with orders
customers = static_reference_data['customers'].copy()

# Menu items for all restaurants: for matching plant-based labels with orders
items_tagged = static_reference_data['items_tagged'].copy()

# Restaurant details (30 rows)
locations = static_reference_data['locations'].copy()

(Formatting to be moved to processing notebook) Set the index as the restaurant IDs because there are only 30 rows

In [ ]:
# Check if location_id is a column to prevent attempting to a remove a column that isn't there
if 'location_id' in before_after_details.columns:
    before_after_details.index = before_after_details['location_id']
    before_after_details.drop(columns=['location_id'], inplace=True)

# Check if location_id is a column to prevent attempting to a remove a column that isn't there
if 'location_id' in locations.columns:
    locations.index = locations['location_id']
    locations.drop(columns=['location_id'], inplace=True)

### Menu Stats

General

In [ ]:
items_tagged.columns.tolist()

In [ ]:
items_tagged['item_type'].value_counts()

In [ ]:
items_tagged['dish_category'].value_counts()

In [ ]:
items_tagged['ingredients'].value_counts()

In [ ]:
items_tagged['brand'].value_counts()

Specific

In [ ]:
# Create a filter dataframe for easier filtering
items_tagged_fdf = fdf(items_tagged)

# Filter out alcohol
items_tagged_no_alcohol = items_tagged_fdf.filter('dish_category', 'Alcohol', exclude=True)
items_tagged_no_alcohol_fdf = fdf(items_tagged_no_alcohol)

plant_based = items_tagged_no_alcohol_fdf.filter('is_plant_based', 'yes')

### Menu Visuals

In [ ]:
plt.bar(items_tagged['is_plant_based'].value_counts().index, items_tagged['is_plant_based'].value_counts())
plt.title("Total Menu Items: Is it Plant-Based?")
plt.show()

In [ ]:
plt.bar(items_tagged['item_type'].value_counts().index, items_tagged['item_type'].value_counts())
plt.title("Total Menu Items: Item Types")
plt.show()

In [ ]:
plt.bar(items_tagged['dish_category'].value_counts()[:20].index, items_tagged['dish_category'].value_counts()[:20])
plt.title("Total Menu Items: Dish Categories")
plt.xticks(rotation = 75)
plt.show()

In [ ]:
plt.bar(items_tagged_no_alcohol['is_plant_based'].value_counts().index, items_tagged_no_alcohol['is_plant_based'].value_counts(), color="red")
plt.title("Menu Items without Alcohol: Is it Plant-Based?")
plt.show()

In [ ]:
plt.bar(items_tagged_no_alcohol['item_type'].value_counts().index, items_tagged_no_alcohol['item_type'].value_counts(), color="red")
plt.title("Menu Items without Alcohol: Item Types")
plt.show()

In [ ]:
plt.bar(items_tagged_no_alcohol['dish_category'].value_counts()[:20].index, items_tagged_no_alcohol['dish_category'].value_counts()[:20], color="red")
plt.title("Menu Items without Alcohol: Dish Categories")
plt.xticks(rotation = 75)
plt.show()

In [ ]:
plt.bar(plant_based['item_type'].value_counts().index, plant_based['item_type'].value_counts(), color="green")
plt.title("Plant-Based Menu Items without Alcohol: Item Types")
plt.show()

In [ ]:
plt.bar(plant_based['dish_category'].value_counts()[:20].index, plant_based['dish_category'].value_counts()[:20], color="green")
plt.title("Plant-Based Menu Items without Alcohol: Dish Categories")
plt.xticks(rotation = 75)
plt.show()

## Restaurant Sales Data Exploration

### Retrieval

Retrieve the location ids

In [ ]:
location_ids = list(restaurant_sales_data.keys())

### General

Columns

In [ ]:
restaurant_sales_data[location_ids[0]].columns.tolist()

In [ ]:
fdf(restaurant_sales_data[location_ids[0]]).filter('item_name','Fries')

### Order Stats

Timeframes

In [ ]:
list_of_timeframes = []

# Find the time difference
for location_id, df in tqdm(restaurant_sales_data.items()):

    # Find the time difference
    timedelta = df.index[-1] - df.index[0]

    # Convert to days, then to years
    years = timedelta.days / 365.25

    # Append
    list_of_timeframes.append(years)

# Turn into an np array for finding median, mean, and std
timeframes = np.array(list_of_timeframes)

# Display
print("Median: {:.2f} year range".format(np.median(timeframes)))
print("Mean: {:.2f} year range".format(np.mean(timeframes)))
print("SD: {:.2f} years".format(np.std(timeframes)))
print("Restaurants with less than a 2 year range: {}".format((timeframes < 2).sum()))

### Data Integrity Checks

Menu data item name per restaurant (non)uniqueness

In [ ]:
items_tagged['id'].duplicated().sum(), items_tagged[['location_id', 'item_name']].duplicated().sum(), items_tagged[['location_id', 'item_name']].duplicated(keep=False).sum()

In [ ]:
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_rows', 500)

In [ ]:
items_tagged[items_tagged[['location_id', 'item_name']].duplicated(keep=False)].sort_values(['location_id','item_name']).head(2)

Sales data item name and unit price (non)uniqueness

In [ ]:
def find_duplicate_sales(df, location_id):
    """
    Identifies sales rows that are exact duplicates in a restaurant sales dataframe.

    Args:
    df_ (DataFrame): Input DataFrame containing sales data.
    location_id: Identifier for the location.

    Returns:
    Tuple[DataFrame, list]: A tuple containing a DataFrame of near duplicates and a list of summaries.
    """

    # Copy to prevent overwriting
    df = df.copy()
    
    # Count exact duplicates
    duplicates = df[df.duplicated(keep=False)]
    duplicates.columns.name = location_id
    exact_duplicate_count = duplicates.drop_duplicates().shape[0]
    exact_duplicate_total = df.duplicated().sum()

    # Count duplicates by id
    ids = df['unique_id']
    duplicate_ids = ids[ids.duplicated(keep=False)]
    id_duplicate_count = duplicate_ids.nunique()
    id_duplicate_total = ids.duplicated().sum()
    id_list = duplicate_ids.drop_duplicates().tolist()

    # Are there non exact duplicates? No
    non_exact_duplicate_bool = exact_duplicate_count != id_duplicate_count and exact_duplicate_total != id_duplicate_total
    if non_exact_duplicate_bool:
        
        # Count the number of unique values in each column per group
        nunique_per_group = df.groupby('unique_id').nunique()
    
        # Identify unique_ids with more than one unique value in any column
        non_unique_ids = nunique_per_group.max(axis=1) > 1
        non_unique_ids = non_unique_ids[non_unique_ids].index
    
        # Exclude groups where all rows are identical
        total_rows_per_group = df.groupby('unique_id').size()
        non_identical_groups = total_rows_per_group[non_unique_ids] != nunique_per_group.loc[non_unique_ids].max(axis=1)
        filtered_unique_ids = non_identical_groups[non_identical_groups].index
    
    # Prepare a row of summary values
    summary = {
        'location_id': location_id,
        'rows_delivered': df.shape[0],
        'non_exact_id_duplicates': id_duplicate_total - exact_duplicate_total,
        'exact_duplicate_count': exact_duplicate_count,
        'exact_duplicate_total': exact_duplicate_total,
        'unique_rows': df.shape[0] - exact_duplicate_total
    }
    
    return duplicates, id_list, summary

In [ ]:
def find_duplicate_items(df_, location_id):
    """
    Identifies items with distinct unit prices in a restaurant sales dataframe.

    Args:
    df_ (DataFrame): Input DataFrame containing sales data.
    location_id: Identifier for the location.

    Returns:
    Tuple[DataFrame, list]: A tuple containing a DataFrame of items with distinct unit prices and a list of summaries.
    """
    
    # Copy to prevent overwriting
    df = df_.copy()

    # Define the threshold for considering prices as distinct (in this case, 2 cents)
    threshold = 2

    # Filter out rows where 'item_name' is NaN
    df = df[~(df['item_name'] == 'nan')]
    
    # Find duplicate unit prices and near-duplicate unit prices and remove
    df.sort_values(['item_name', 'unit_price'], inplace=True) # sort so that unit prices
    df['price_diff'] = df.groupby('item_name')['unit_price'].diff().abs() # within each item name, take
    distinct_unit_prices_data = df[(df['price_diff'].isna() | (df['price_diff'] >= threshold))] # the first item will always be NA so include that
    
    # Finding the actual items that have distinct unit prices since items with one unit price are included by default
    distinct_unit_price_counts = distinct_unit_prices_data.groupby('item_name')['unit_price'].nunique()
    items_with_distinct_prices = distinct_unit_price_counts[distinct_unit_price_counts > 1].index
    item_list = items_with_distinct_prices.tolist()
    data = distinct_unit_prices_data[distinct_unit_prices_data['item_name'].isin(items_with_distinct_prices)]
    data.columns.name = location_id
    
    # Determine which items have multiple prices (with price differences greater than 2 cents) and filter to only those
    distinct_unit_prices_count = data['item_name'].nunique()
    distinct_unit_prices_total = data.shape[0] - distinct_unit_prices_count

    # Prepare a row of summary values
    summary = {'location_id' : location_id, 
               'distinct_unit_prices_count' : distinct_unit_prices_count, 
               'distinct_unit_prices_total' : distinct_unit_prices_total
    }    
    
    return data, item_list, summary

In [ ]:
# Prepare lists for exact duplicate sales data and a summary of it
duplicate_sales_list = []
id_dict = {}
duplicate_sales_summary_list = []


# Prepare lists for duplicate item names with different unit prices data and a summary of it
duplicate_items_list = []
name_dict = {}
duplicate_items_summary_list = []

# For every restaurant
for location_id, df in tqdm(restaurant_sales_data.items()):

    # Compute the exact duplicate sales entries
    duplicate_sales_one_restaurant, id_list, duplicate_sales_summary_one_restaurant = find_duplicate_sales(df, location_id)
    duplicate_sales_list.append(duplicate_sales_one_restaurant)
    id_dict[location_id] = id_list
    duplicate_sales_summary_list.append(duplicate_sales_summary_one_restaurant)

    # Compute the duplicate items with distinct unit prices
    duplicate_items_one_restaurant, name_list, duplicate_items_summary_one_restaurant = find_duplicate_items(df, location_id)
    duplicate_items_list.append(duplicate_items_one_restaurant)
    name_dict[location_id] = name_list
    duplicate_items_summary_list.append(duplicate_items_summary_one_restaurant)

# Create summary dataframes
duplicate_sales_summary = pd.DataFrame(duplicate_sales_summary_list)
duplicate_items_summary = pd.DataFrame(duplicate_items_summary_list)

In [ ]:
duplicate_sales_summary

In [ ]:
duplicate_sales_summary['rows_delivered'].sum(), duplicate_sales_summary['unique_rows'].sum()

Expected and Received

In [ ]:
duplicate_sales_summary['rows_delivered'].sum(), duplicate_sales_summary['unique_rows'].sum()

Exact Duplicates Sales (too large, so don't export)

In [ ]:
# Write results to an Excel file
# if not os.path.exists('example_palate_data_issues/exact_duplicate_sales.xlsx'):
#     with pd.ExcelWriter('example_palate_data_issues/exact_duplicate_sales.xlsx') as writer:
#         duplicate_sales_summary.to_excel(writer, sheet_name='summary')
#         for df_ in tqdm(duplicate_sales_list):
#             df = df_.copy()
#             df.index = df.index.astype(str)
#             df.to_excel(writer, sheet_name=df.columns.name)

Near Duplicate Items with Distinct Unit Prices

In [ ]:
# Write results to an Excel file
if not os.path.exists('example_palate_data_issues/near_duplicate_items.xlsx'):
    with pd.ExcelWriter('example_palate_data_issues/near_duplicate_items.xlsx') as writer:
        duplicate_items_summary.to_excel(writer, sheet_name='summary')
        for df_ in tqdm(duplicate_items_list):
            df = df_.copy()
            df.index = df.index.astype(str)
            df.to_excel(writer, sheet_name=df.columns.name)

Fractional item quantities

In [ ]:
fractional_quantity_restaurant_ids = []
for loc_id, df in tqdm(restaurant_sales_data.items()):
    if not df[~df['item_quantity'].astype(str).str.contains(".0")].empty:
        fractional_quantity_restaurant_ids.append(loc_id)

In [ ]:
# Write results to an Excel file
if not os.path.exists('example_palate_data_issues/fractional_quantities.xlsx'):
    with pd.ExcelWriter('example_palate_data_issues/fractional_quantities.xlsx') as writer:
        for location_id in fractional_quantity_restaurant_ids:
            df = restaurant_sales_data[location_id].copy()
            df.index = df.index.astype(str)
            fractional_quantity_examples = df[~df['item_quantity'].astype(str).str.contains(".0")]
            fractional_quantity_examples.to_excel(writer, sheet_name=location_id)

Differently Capitalized Duplicates

In [ ]:
my_fdf = fdf(restaurant_sales_data['1SQPTEGYPH0GA'])
antipasti = my_fdf.filter('item_name', 'antipasti').reset_index().groupby('unit_price').min().sort_values('created_at')

In [ ]:
# # Write results to an Excel file
# if not os.path.exists('example_palate_data_issues/same_item_different_unit_prices.xlsx'):
#     with pd.ExcelWriter('example_palate_data_issues/same_item_different_unit_prices.xlsx') as writer:
#         antipasti.to_excel(writer)

In [ ]:
capitalization_discrepancies = []
for loc_id, df in restaurant_sales_data.items():

    # Determine how many unique items there are if you force a certain capitalization
    capitalization_discrepancies.append((loc_id, df['item_name'].unique().size, df['item_name'].str.lower().unique().size))

# Make into a df and label
cap_discrepancies = pd.DataFrame(capitalization_discrepancies)
cap_discrepancies.columns = ['location', 'caps_sens', 'not_caps_sens']
cap_discrepancies['discrepancy'] = cap_discrepancies['caps_sens'] != cap_discrepancies['not_caps_sens']

#cap_discrepancies

In [ ]:
#my_fdf = fdf(restaurant_sales_data['1SQPTEGYPH0GA'])
#my_fdf.filter('item_name', 'antipasti')

### Data Integrity Fixes

#### Static Reference Data

Drop duplicates within each restaurant

In [ ]:
# For now, drop duplicates (items that have same name within a single restaurant)
items_tagged.drop_duplicates(['location_id', 'item_name'], inplace=True)

#### Restaurant Sales Data

Standardize capitalization

In [ ]:
for loc_id, df in restaurant_sales_data.items():
    df['item_name'] = df['item_name'].str.capitalize()

Drop exact duplicates

In [ ]:
# Some the data is exactly duplicated
for location_id, df in restaurant_sales_data.items():
    df.drop_duplicates('unique_id', inplace=True)

### Merging Data

Merging the sales data and the menu data

In [ ]:
# Check if 'items_tagged' has unique pairs of 'item_name' and 'location_id'
if items_tagged.duplicated(subset=['item_name', 'location_id']).any():
    raise ValueError("Duplicates found in 'items_tagged' for the combination of 'item_name' and 'location_id'")

merged_sales_and_menu = {}
for location_id, df in tqdm(restaurant_sales_data.items()):

    # Copy to prevent overwriting
    df_copy = df.copy()
    items_tagged_copy = items_tagged.copy()

    # For ease of merging, pretend items of different capitalizations are the same <-------- *Note to potentially remove later*
    df_copy['item_name'] = df_copy['item_name'].str.lower()
    items_tagged_copy['item_name'] = items_tagged_copy['item_name'].str.lower()
    items_tagged_copy.drop_duplicates(['item_name', 'location_id'], inplace=True)
    
    # Perform the merge on both 'item_name' and 'location_id'
    merged = pd.merge(df_copy.reset_index(), items_tagged_copy, 
                      on=['item_name', 'location_id'], how='left')
    
    # Set 'created_at' back as the index
    merged.set_index('created_at', inplace=True)

    # Remove failed merges due to missing data on 27 and encoding errors elsewhere <-------- *Note to potentially remove later*
    merged = merged[~merged['is_plant_based'].isna()]

    # Recapitalize
    merged['item_name'].str.capitalize()

    # Store in the dictionary
    merged_sales_and_menu[location_id] = merged

In [ ]:
# for i in range(30):
#     print(merged_sales_and_menu[location_ids[i]][merged_sales_and_menu[location_ids[i]]['is_plant_based'].isna()]['item_name'].unique())

In [ ]:
# Example of problem 1, missing menu data for 27
# fdf(items_tagged).multi_filter([('location_id', location_ids[27])])['item_name'].str.lower().sort_values().iloc[50:].head(20)

# Example of problem 2, encoding error for multiple restaurants
# fdf(items_tagged).multi_filter([('location_id', location_ids[2])])['item_name'].str.lower().sort_values().iloc[622:].head(20)
# fdf(items_tagged).filter('item_name', 'kahl√∫a')

### Benchmarks

Pareto analysis

In [ ]:
list_of_percentiles = []
list_of_num_dishes = []

# Determine the number of menu items for all 30 restaurants
for loc_id in location_ids:

    # Filter to the current restaurant
    menu_items = items_tagged_fdf.filter('location_id', loc_id).drop_duplicates('id')


    sales_and_menu_no_alcohol = fdf(merged_sales_and_menu[loc_id]).filter('dish_category', 'Alcohol', exclude=True)
    
    # 'entries', 'quantity', 'sales'
    target_percentage = .8
    percentiles = pa.create_percentiles(sales_and_menu_no_alcohol, metric='quantity')
    index = pa.num_items_to_cover_certain_percentage(sales_and_menu_no_alcohol, metric='quantity', percentile=target_percentage)
    
    list_of_percentiles.append(percentiles)
    list_of_num_dishes.append((index+1, percentiles.size))

    top_items = percentiles.iloc[:index+1]

    items_tagged_copy = items_tagged.copy()
    menu_no_alcohol_all = fdf(items_tagged_copy).filter('dish_category', 'Alcohol', exclude=True)
    menu_no_alcohol = fdf(menu_no_alcohol_all).filter('location_id', loc_id)
    menu_no_alcohol['item_name'] = menu_no_alcohol['item_name'].str.lower()
    menu_no_alcohol.drop_duplicates('item_name', inplace=True)
    menu_no_alcohol['item_name'] = menu_no_alcohol['item_name'].str.capitalize()

    sample = ac.calculate_accuracy(top_items, menu_no_alcohol, n_sample=10).loc[:,['item_name', 'item_type', 'dish_category', 'is_plant_based', 'ingredients']]


In [ ]:
pd.DataFrame(list_of_num_dishes)[0].sum()

heuristics used:
- raspberry cookies are not plantbased
- pickle chips are unclear (they said yes)
- eggplant sandwhich is not plantbased because the ingredient was given as mozzarella (opens question about ingredients)
- eggplant side without ingredient is plant based
- ginger miso listed as dumplings with no ingredients is ambigious even though they said no
- garlic fries are unclear (they said no)
- char siu was given a false positive since it was listed as unsure...
- we'll call impossible melt plant based... (restaurant 8)
- Restaurant 9 has alcohol that isnt in the alcohol section
- Restaurant 12 has a milk latte, shouldn't be plant based (noted in ingredients)
- To reiterate, things marked as unsure when they are clearly meat get a false positive



In [ ]:
sample_mislabel_list = [
{'restaurant_id':location_ids[0],
'true_positives': 3, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 1,
'ambiguous': 1},

{'restaurant_id':location_ids[1],
'true_positives': 1, 
'false_positives': 0,
'true_negatives': 4,
'false_negatives': 2,
'ambiguous': 3},

{'restaurant_id':location_ids[2],
'true_positives': 1, 
'false_positives': 2,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 5},

{'restaurant_id':location_ids[3],
'true_positives': 1, 
'false_positives': 2,
'true_negatives': 4,
'false_negatives': 0,
'ambiguous': 3},

{'restaurant_id':location_ids[4],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 0,
'ambiguous': 1},

{'restaurant_id':location_ids[5],
'true_positives': 2, 
'false_positives': 1,
'true_negatives': 4,
'false_negatives': 0,
'ambiguous': 3},

{'restaurant_id':location_ids[6],
'true_positives': 1, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 7},   # I have no idea what this stuff is

{'restaurant_id':location_ids[7],
'true_positives': 1, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 7},

{'restaurant_id':location_ids[8],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 3,
'false_negatives': 0,
'ambiguous': 3},

{'restaurant_id':location_ids[9],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 1,
'false_negatives': 5,
'ambiguous': 0},

{'restaurant_id':location_ids[10],
'true_positives': 7, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 1},

{'restaurant_id':location_ids[11],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 4},

{'restaurant_id':location_ids[12],
'true_positives': 7,    # probably more like ambigious given coffee usually has creamer
'false_positives': 2,
'true_negatives': 0,
'false_negatives': 0,
'ambiguous': 1},

{'restaurant_id':location_ids[13],
'true_positives': 2, 
'false_positives': 1,
'true_negatives': 7,
'false_negatives': 0,
'ambiguous': 0},

{'restaurant_id':location_ids[14],
'true_positives': 5, 
'false_positives': 1,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 2},

{'restaurant_id':location_ids[15],
'true_positives': 2, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 1,
'ambiguous': 2},

{'restaurant_id':location_ids[16],
'true_positives': 3, 
'false_positives': 0,
'true_negatives': 6,
'false_negatives': 0,
'ambiguous': 1},

{'restaurant_id':location_ids[17],
'true_positives': 0, 
'false_positives': 1,
'true_negatives': 7,
'false_negatives': 0,
'ambiguous': 2},

{'restaurant_id':location_ids[18],
'true_positives': 2, 
'false_positives': 1,
'true_negatives': 4,
'false_negatives': 2,
'ambiguous': 1},

{'restaurant_id':location_ids[19],
'true_positives': 2, 
'false_positives': 3,
'true_negatives': 4,
'false_negatives': 0,
'ambiguous': 1},   # credit card fee?

{'restaurant_id':location_ids[20],
'true_positives': 3, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 5},  # "custom amount"?

{'restaurant_id':location_ids[21],
'true_positives': 1, 
'false_positives': 0,
'true_negatives': 6,
'false_negatives': 1,
'ambiguous': 2},

{'restaurant_id':location_ids[22],    # only 8 items for top 90%
'true_positives': 3, # we'll count impossible melt as yes
'false_positives': 0,
'true_negatives': 4,
'false_negatives': 0,
'ambiguous': 1},  # maybe 2 ambigious if we count gold standard kale with egg ingredient

{'restaurant_id':location_ids[23],
'true_positives': 4,  # sausage, egg, cheese (v) ? 
'false_positives': 3,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 1},

{'restaurant_id':location_ids[24],
'true_positives': 4, 
'false_positives': 1,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 3},

{'restaurant_id':location_ids[25],
'true_positives': 4, 
'false_positives': 1,
'true_negatives': 5,
'false_negatives': 0,
'ambiguous': 0},

{'restaurant_id':location_ids[26],  # only has like 10 items in top 90%
'true_positives': 2, 
'false_positives': 0,
'true_negatives': 4,
'false_negatives': 3,
'ambiguous': 1},  # "single"

{'restaurant_id':location_ids[27],   # more alcohol not labeled in alcohol category
'true_positives': 6, 
'false_positives': 0,
'true_negatives': 1,
'false_negatives': 0, 
'ambiguous': 3},   # "egift card", "foreland stolen artifacts"

{'restaurant_id':location_ids[28],
'true_positives': 2, 
'false_positives': 3,
'true_negatives': 1,
'false_negatives': 0,
'ambiguous': 4},  # "card"

{'restaurant_id':location_ids[29],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 0,
'ambiguous': 1},
]

In [ ]:
sample_mislabel_counts = pd.DataFrame(sample_mislabel_list)

In [ ]:
accuracy = sample_mislabel_counts[['true_positives', 'true_negatives']].sum().sum() / 300
error_rate = sample_mislabel_counts[['false_positives', 'false_negatives']].sum().sum() / 300
ambiguous = sample_mislabel_counts['ambiguous'].sum() / 300
sensitivity = sample_mislabel_counts['true_positives'].sum() / sample_mislabel_counts[['true_positives', 'false_negatives']].sum().sum()
specificity = sample_mislabel_counts['true_negatives'].sum() / sample_mislabel_counts[['false_positives', 'true_negatives']].sum().sum()
precision = sample_mislabel_counts['true_positives'].sum() / sample_mislabel_counts[['true_positives', 'false_positives']].sum().sum()
negative_predictive_value = sample_mislabel_counts['true_negatives'].sum() / sample_mislabel_counts[['true_negatives', 'false_negatives']].sum().sum()



"General split:", accuracy, error_rate, ambiguous, "More details: ", sensitivity, specificity, precision, negative_predictive_value

Fixing capitalization errors in merging

In [ ]:
# Too difficult, just lower everything for counting errors

# merge_mismatches= [] 
# for loc_id in tqdm(location_ids):
#     restaurant_df = fdf(items_tagged).filter('location_id', loc_id)
#     items = restaurant_df['item_name'].tolist()
#     restaurant_merge_mismatches_list = []
#     for item in items:
#         capitalization_variation_counts = fdf(restaurant_df).filter('item_name', item)
#         restaurant_merge_mismatches_list.append((loc_id, item, capitalization_variation_counts.shape[0] - 1))
#         restaurant_merge_mismatches = pd.DataFrame(restaurant_merge_mismatches_list)
#     merge_mismatches.append((loc_id, restaurant_merge_mismatches[2].any()))



Accuracy calculation

### Promotional Items

In [ ]:
for location_id, df in list(merged_sales_and_menu.items()):
    promotional_item = before_after_details.loc[location_id,'first_plant_based_mention']
    promo_df = fdf(df).filter('item_name', promotional_item)
    #print(promotional_item)
    #print(promo_df['item_name'].head(1))

Identifying first plant-based

In [ ]:
for location_id, df in merged_sales_and_menu.items():
    plant_based = fdf(df).filter('is_plant_based', 'yes')
    #print(before_after_details.loc[location_id,'first_plant_based_mention'])
    #print(plant_based['item_name'].head(10))

### Sales Visuals

In [ ]:
num_plots = len(merged_sales_and_menu)
cols = 4 
rows = math.ceil(num_plots / cols) * 4

# Create a figure with multiple subplots
fig, axs = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axs = axs.flatten()  # Flatten the array for easy indexing

top_dishes_number = 20

for i, (location_id, data) in tqdm(enumerate(merged_sales_and_menu.items())):

    df = data.copy()
    
    # Get the top 20 items sorted in descending order
    top_items_by_times_ordered = df['item_name'].value_counts().nlargest(top_dishes_number).sort_values()

    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i].barh(top_items_by_times_ordered.index.str.slice(0,20).str.capitalize(), top_items_by_times_ordered)
    axs[4*i].set_title(f'{location_id}\nTotal by Times Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i].tick_params(axis='y', labelsize=10)
    axs[4*i].set_xlabel('Times Ordered')

    top_items_by_quantity_ordered = df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values()
    
    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 1].barh(top_items_by_quantity_ordered.index.str.slice(0,20).str.capitalize(), top_items_by_quantity_ordered, color='cyan')
    axs[4*i + 1].set_title(f'{location_id}\nTotal by Quantity Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 1].tick_params(axis='y', labelsize=10)
    axs[4*i + 1].set_xlabel('Quantity Ordered')

    plant_df = df[df['is_plant_based'] == 'yes']
    
    # Get the top 20 items sorted in descending order
    top_plant_based_items_by_times_ordered = plant_df['item_name'].value_counts().nlargest(top_dishes_number).sort_values()

    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 2].barh(top_plant_based_items_by_times_ordered.index.str.slice(0,20).str.capitalize(), top_plant_based_items_by_times_ordered, color='green')
    axs[4*i + 2].set_title(f'{location_id}\nPlant-Based by Times Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 2].tick_params(axis='y', labelsize=10)
    axs[4*i + 2].set_xlabel('Times Ordered')

    top_plant_based_items_by_quantity_ordered = plant_df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values()
    
    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 3].barh(top_plant_based_items_by_quantity_ordered.index.str.slice(0,20).str.capitalize(), top_plant_based_items_by_quantity_ordered, color='lime')
    axs[4*i + 3].set_title(f'{location_id}\nPlant-Based by Quantity Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 3].tick_params(axis='y', labelsize=10)
    axs[4*i + 3].set_xlabel('Quantity Ordered')

# Add a global title at the top of the figure
fig.suptitle('Distribution of Plant-Based and Total Item Orders in Each Restaurant', fontsize=16)

# Adjust layout and save the figure
plt.tight_layout()
plt.subplots_adjust(top=0.95)  # Adjust the top margin to make room for the global title
plt.savefig('Restaurant Both Sales Distributions.png', bbox_inches='tight')